# In-Memory Vector Store Reference

Developer-facing statements defined in `langchain_core.vectorstores.in_memory`.

# `InMemoryVectorStore: VectorStore`

Dictionary-backed vector store that embeds documents and performs cosine-similarity search in memory.

It supports synchronous and asynchronous document addition, ID lookup, deletion, similarity search, metadata-aware filtering, maximal marginal relevance search, and JSON persistence.

## Fields

```python
store: dict[str, dict[str, Any]] # Stored entries keyed by document ID
embedding: Embeddings # Embedding implementation used for indexing and queries
```

Each value in `store` contains `"id"`, `"vector"`, `"text"`, and `"metadata"`.

## Constructor

```python
InMemoryVectorStore(
    embedding: Embeddings, # Embedding implementation used by the store
) -> None
```

## Properties

### `embeddings`

Returns the configured embedding implementation.

```python
embeddings: Embeddings
```

## Document mutation

### `add_documents`

Embeds and stores documents synchronously.

```python
add_documents(
    self,
    documents: list[Document], # Documents to add or replace
    ids: list[str] | None = None, # IDs overriding IDs stored on the documents
    **kwargs: Any, # Additional arguments accepted for interface compatibility
) -> list[str] # IDs assigned to the stored documents
```

Explicit `ids` take precedence over `Document.id`. When neither is available, a UUID is generated. Adding an existing ID replaces its stored entry.

Raises `ValueError` when a non-empty `ids` list has a different length from `documents`.

### `aadd_documents`

Asynchronously embeds and stores documents using `aembed_documents`.

```python
async aadd_documents(
    self,
    documents: list[Document], # Documents to add or replace
    ids: list[str] | None = None, # IDs overriding IDs stored on the documents
    **kwargs: Any, # Additional arguments accepted for interface compatibility
) -> list[str] # IDs assigned to the stored documents
```

### `delete`

Removes entries matching the supplied IDs. Missing IDs are ignored.

```python
delete(
    self,
    ids: Sequence[str] | None = None, # IDs to remove
    **kwargs: Any, # Additional arguments accepted for interface compatibility
) -> None
```

Passing `None` or an empty sequence performs no deletion.

### `adelete`

Asynchronous deletion wrapper that calls `delete`.

```python
async adelete(
    self,
    ids: Sequence[str] | None = None, # IDs to remove
    **kwargs: Any, # Additional arguments accepted for interface compatibility
) -> None
```

## ID lookup

### `get_by_ids`

Returns stored documents for the requested IDs.

```python
get_by_ids(
    self,
    ids: Sequence[str], # IDs to retrieve
    /,
) -> list[Document] # Documents found for the supplied IDs
```

Missing IDs are ignored. Results follow the supplied ID order, and repeated IDs can produce repeated documents.

### `aget_by_ids`

Asynchronous wrapper around `get_by_ids`.

```python
async aget_by_ids(
    self,
    ids: Sequence[str], # IDs to retrieve
    /,
) -> list[Document] # Documents found for the supplied IDs
```

## Similarity search

### `similarity_search_with_score_by_vector`

Searches using a supplied embedding vector and returns cosine-similarity scores.

```python
similarity_search_with_score_by_vector(
    self,
    embedding: list[float], # Query embedding vector
    k: int = 4, # Maximum number of results
    filter: Callable[[Document], bool] | None = None, # Optional document predicate applied before ranking
    **_kwargs: Any, # Additional ignored search arguments
) -> list[tuple[Document, float]] # Documents paired with cosine-similarity scores
```

Higher scores indicate greater similarity.

### `similarity_search_with_score`

Embeds the query synchronously and delegates to vector-based scored search.

```python
similarity_search_with_score(
    self,
    query: str, # Query text
    k: int = 4, # Maximum number of results
    **kwargs: Any, # Arguments forwarded to vector-based scored search
) -> list[tuple[Document, float]] # Documents paired with cosine-similarity scores
```

### `asimilarity_search_with_score`

Embeds the query asynchronously and performs scored vector search.

```python
async asimilarity_search_with_score(
    self,
    query: str, # Query text
    k: int = 4, # Maximum number of results
    **kwargs: Any, # Arguments forwarded to vector-based scored search
) -> list[tuple[Document, float]] # Documents paired with cosine-similarity scores
```

### `similarity_search_by_vector`

Returns documents most similar to a supplied embedding vector.

```python
similarity_search_by_vector(
    self,
    embedding: list[float], # Query embedding vector
    k: int = 4, # Maximum number of results
    **kwargs: Any, # Arguments forwarded to vector-based scored search
) -> list[Document] # Most similar documents
```

### `asimilarity_search_by_vector`

Asynchronous wrapper around `similarity_search_by_vector`.

```python
async asimilarity_search_by_vector(
    self,
    embedding: list[float], # Query embedding vector
    k: int = 4, # Maximum number of results
    **kwargs: Any, # Arguments forwarded to vector-based search
) -> list[Document] # Most similar documents
```

### `similarity_search`

Embeds the query and returns the highest-scoring documents.

```python
similarity_search(
    self,
    query: str, # Query text
    k: int = 4, # Maximum number of results
    **kwargs: Any, # Arguments forwarded to scored search
) -> list[Document] # Most similar documents
```

### `asimilarity_search`

Asynchronously embeds the query and returns the highest-scoring documents.

```python
async asimilarity_search(
    self,
    query: str, # Query text
    k: int = 4, # Maximum number of results
    **kwargs: Any, # Arguments forwarded to asynchronous scored search
) -> list[Document] # Most similar documents
```

## Maximal marginal relevance

### `max_marginal_relevance_search_by_vector`

Selects results by balancing similarity to a supplied vector with diversity among selected documents.

```python
max_marginal_relevance_search_by_vector(
    self,
    embedding: list[float], # Query embedding vector
    k: int = 4, # Number of documents to select
    fetch_k: int = 20, # Number of similarity candidates considered
    lambda_mult: float = 0.5, # Balance from maximum diversity at `0` to maximum relevance at `1`
    *,
    filter: Callable[[Document], bool] | None = None, # Optional predicate applied before candidate selection
    **kwargs: Any, # Additional arguments accepted for interface compatibility
) -> list[Document] # Documents selected by maximal marginal relevance
```

Raises `ImportError` when NumPy is unavailable.

### `max_marginal_relevance_search`

Embeds the query synchronously and delegates to vector-based MMR search.

```python
max_marginal_relevance_search(
    self,
    query: str, # Query text
    k: int = 4, # Number of documents to select
    fetch_k: int = 20, # Number of similarity candidates considered
    lambda_mult: float = 0.5, # Relevance-diversity balance
    **kwargs: Any, # Arguments forwarded to vector-based MMR search
) -> list[Document] # Documents selected by maximal marginal relevance
```

### `amax_marginal_relevance_search`

Embeds the query asynchronously and performs vector-based MMR search.

```python
async amax_marginal_relevance_search(
    self,
    query: str, # Query text
    k: int = 4, # Number of documents to select
    fetch_k: int = 20, # Number of similarity candidates considered
    lambda_mult: float = 0.5, # Relevance-diversity balance
    **kwargs: Any, # Arguments forwarded to vector-based MMR search
) -> list[Document] # Documents selected by maximal marginal relevance
```

## Factory methods

### `from_texts`

Creates a store and adds the supplied texts synchronously.

```python
@classmethod
from_texts(
    cls,
    texts: list[str], # Texts used to initialize the store
    embedding: Embeddings, # Embedding implementation to use
    metadatas: list[dict[str, Any]] | None = None, # Optional metadata for each text
    **kwargs: Any, # Arguments forwarded to `add_texts`
) -> InMemoryVectorStore # Initialized in-memory vector store
```

### `afrom_texts`

Creates a store and adds the supplied texts asynchronously.

```python
@classmethod
async afrom_texts(
    cls,
    texts: list[str], # Texts used to initialize the store
    embedding: Embeddings, # Embedding implementation to use
    metadatas: list[dict[str, Any]] | None = None, # Optional metadata for each text
    **kwargs: Any, # Arguments forwarded to `aadd_texts`
) -> InMemoryVectorStore # Initialized in-memory vector store
```

## Persistence

### `load`

Loads a previously dumped store from a UTF-8 JSON file and attaches the supplied embedding implementation.

```python
@classmethod
load(
    cls,
    path: str, # JSON file containing the dumped store
    embedding: Embeddings, # Embedding implementation to attach
    **kwargs: Any, # Additional constructor arguments
) -> InMemoryVectorStore # Restored in-memory vector store
```

### `dump`

Serializes the current store to a UTF-8 JSON file.

```python
dump(
    self,
    path: str, # Destination JSON file
) -> None
```

Missing parent directories are created automatically.

In [ ]:
%pip install -U langchain-core numpy # Install LangChain Core and NumPy

import re # Import regular expressions for extracting words
from langchain_core.documents import Document # Import the LangChain Document class
from langchain_core.embeddings import Embeddings # Import the embedding interface
from langchain_core.vectorstores import InMemoryVectorStore # Import the in-memory vector store


class KeywordEmbeddings(Embeddings): # Create a predictable keyword-based embedding model

    def __init__(self, vocabulary: list[str]) -> None: # Initialize the embedding model
        self.vocabulary = [word.lower() for word in vocabulary] # Store vocabulary words in lowercase

    def _create_vector(self, text: str) -> list[float]: # Convert text into a numeric vector
        words = re.findall(r"\b\w+\b", text.lower()) # Extract lowercase words from the text
        return [float(words.count(word)) for word in self.vocabulary] # Count every vocabulary word

    def embed_documents(self, texts: list[str]) -> list[list[float]]: # Embed multiple documents
        return [self._create_vector(text) for text in texts] # Return one vector for every document

    def embed_query(self, text: str) -> list[float]: # Embed one search query
        return self._create_vector(text) # Return the query vector


vocabulary = [ # Define the meaning of each vector dimension
    "account", # Represent account-related content
    "password", # Represent password-related content
    "login", # Represent login-related content
    "refund", # Represent refund-related content
    "payment", # Represent payment-related content
    "delivery", # Represent delivery-related content
    "order", # Represent order-related content
    "support", # Represent support-related content
] # Finish the vocabulary list

embedding_model = KeywordEmbeddings(vocabulary) # Create the custom embedding model
vector_store = InMemoryVectorStore(embedding=embedding_model) # Create an empty in-memory vector store


documents = [ # Create customer-support FAQ documents
    Document( # Create the password-reset document
        id="faq-1", # Assign a stable document ID
        page_content="Reset your account password from the account settings page.", # Store the FAQ content
        metadata={"category": "account", "priority": "high"}, # Store searchable metadata
    ), # Finish the first document
    Document( # Create the login-support document
        id="faq-2", # Assign a stable document ID
        page_content="Contact support when your account login is not working.", # Store the FAQ content
        metadata={"category": "account", "priority": "high"}, # Store searchable metadata
    ), # Finish the second document
    Document( # Create the refund document
        id="faq-3", # Assign a stable document ID
        page_content="Refunds are returned to the original payment method.", # Store the FAQ content
        metadata={"category": "billing", "priority": "medium"}, # Store searchable metadata
    ), # Finish the third document
    Document( # Create the delivery-tracking document
        id="faq-4", # Assign a stable document ID
        page_content="Track your delivery from the order tracking page.", # Store the FAQ content
        metadata={"category": "delivery", "priority": "medium"}, # Store searchable metadata
    ), # Finish the fourth document
] # Finish the document list


added_ids = vector_store.add_documents(documents=documents) # Embed and store all FAQ documents
print(f"Added IDs: {added_ids}") # Display the assigned document IDs
print(f"Stored entries: {len(vector_store.store)}") # Display the total number of stored entries

In [ ]:
# Similarity search
query = "How can I reset my account password?" # Define the customer question

results = vector_store.similarity_search( # Search using the question text
    query=query, # Supply the customer question
    k=2, # Return the two most similar documents
) # Finish the search call

print("\nSimilarity-search results:") # Display the result heading

for document in results: # Process every retrieved document
    print(f"{document.id}: {document.page_content}") # Display its ID and content

In [ ]:
# Search with cosine-similarity scores: Higher scores indicate greater similarity.
scored_results = vector_store.similarity_search_with_score( # Search and include similarity scores
    query=query, # Supply the customer question
    k=4, # Return up to four results
) # Finish the scored-search call

print("\nScored results:") # Display the scored-result heading

for document, score in scored_results: # Process every document-score pair
    print(f"{score:.3f} -> {document.id}: {document.page_content}") # Display the score and document

In [ ]:
# Metadata-filtered search
def account_filter(document: Document) -> bool: # Define a metadata filter function
    return document.metadata.get("category") == "account" # Keep only account-related documents


filtered_results = vector_store.similarity_search( # Perform filtered similarity search
    query="I cannot access my account.", # Supply the search query
    k=3, # Return up to three documents
    filter=account_filter, # Apply the account-category filter
) # Finish the filtered-search call

print("\nFiltered account results:") # Display the result heading

for document in filtered_results: # Process every filtered document
    print(f"{document.id}: {document.page_content}") # Display its ID and content

In [ ]:
# Search using an embedding vector
query_vector = embedding_model.embed_query("Track my delivery order.") # Create the query vector manually

vector_results = vector_store.similarity_search_by_vector( # Search using the supplied vector
    embedding=query_vector, # Supply the query embedding
    k=2, # Return the two closest documents
) # Finish the vector-search call

print("\nVector-search results:") # Display the result heading

for document in vector_results: # Process every vector-search result
    print(f"{document.id}: {document.page_content}") # Display its ID and content

In [ ]:
# Maximal marginal relevance search: MMR balances relevance with diversity.
mmr_results = vector_store.max_marginal_relevance_search( # Perform MMR search
    query="Help with my account login and password.", # Supply a broader query
    k=3, # Select three final documents
    fetch_k=4, # Consider four similarity candidates
    lambda_mult=0.5, # Balance relevance and diversity equally
) # Finish the MMR-search call

print("\nMMR results:") # Display the result heading

for document in mmr_results: # Process every MMR-selected document
    print(f"{document.id}: {document.page_content}") # Display its ID and content

In [ ]:
# Retrieve documents by ID
selected_documents = vector_store.get_by_ids( # Retrieve specific stored documents
    ["faq-3", "missing-id", "faq-1"] # Include valid and missing document IDs
) # Finish the ID-lookup call

print("\nDocuments retrieved by ID:") # Display the result heading

for document in selected_documents: # Process every document that was found
    print(f"{document.id}: {document.page_content}") # Display its ID and content

In [ ]:
# Convert the store into a retriever
retriever = vector_store.as_retriever( # Create a retriever backed by the vector store
    search_type="similarity", # Select ordinary similarity search
    search_kwargs={"k": 2}, # Retrieve two documents by default
) # Finish creating the retriever

retriever_results = retriever.invoke("My account login has failed.") # Retrieve relevant documents

print("\nRetriever results:") # Display the result heading

for document in retriever_results: # Process every retrieved document
    print(f"{document.id}: {document.page_content}") # Display its ID and content

In [ ]:
# Asynchronous operations in Jupyter
new_document = Document( # Create a new document for asynchronous addition
    id="faq-5", # Assign a stable ID
    page_content="Update your payment details from the billing settings.", # Store the FAQ content
    metadata={"category": "billing", "priority": "medium"}, # Store its metadata
) # Finish creating the new document

async_added_ids = await vector_store.aadd_documents([new_document]) # Add the document asynchronously
print(f"\nAsynchronously added IDs: {async_added_ids}") # Display the assigned IDs

async_results = await vector_store.asimilarity_search( # Run similarity search asynchronously
    query="How do I update my payment?", # Supply the asynchronous query
    k=2, # Return two documents
) # Finish the asynchronous search

for document in async_results: # Process every asynchronous result
    print(f"{document.id}: {document.page_content}") # Display its ID and content

In [ ]:
# Save and restore the vector store
file_path = "saved_data/faq_vector_store.json" # Define the JSON persistence path

vector_store.dump(file_path) # Save documents and vectors to the JSON file
print(f"\nVector store saved to: {file_path}") # Display the saved file path

loaded_store = InMemoryVectorStore.load( # Restore the previously saved vector store
    path=file_path, # Supply the JSON file path
    embedding=embedding_model, # Reattach the embedding implementation
) # Finish loading the store

loaded_results = loaded_store.similarity_search( # Search the restored vector store
    query="Where is my refund?", # Supply a query
    k=1, # Return the closest document
) # Finish the restored-store search

print(f"Loaded result: {loaded_results[0].page_content}") # Display the restored search result

In [ ]:
# Delete documents
vector_store.delete(ids=["faq-5", "missing-id"]) # Delete one existing and one missing ID
remaining_document = vector_store.get_by_ids(["faq-5"]) # Check whether the deleted document remains
print(f"\nfaq-5 exists after deletion: {bool(remaining_document)}") # Display the deletion verification